# Notebook 03 — Compliance Checker
**Goal:** Test the two-layer compliance checker (blocklist + LLM classifier) on real and synthetic content.

By the end of this notebook you will have:
- Tested the regex blocklist against known-bad phrases
- Tested the GPT-4o-mini classifier on edge cases
- Run the full checker on actual transcript chunks from your corpus
- Seen how the `check_compliance` agent tool formats its output

**Prerequisite:** Run notebook 01 first to have transcript data in Pinecone.

## Step 1 — Environment check

In [ ]:
import sys
sys.path.append('..')

from src.utils.config import OPENAI_API_KEY, OPENAI_LLM_MODEL

print('✅ OpenAI key loaded:', OPENAI_API_KEY[:8] + '...')
print(f'✅ LLM model: {OPENAI_LLM_MODEL}')

## Step 2 — Inspect the blocklist
See all forbidden phrases that trigger an instant non-compliant verdict.

In [ ]:
from src.compliance.checker import FORBIDDEN_PHRASES

print(f'Total forbidden phrases: {len(FORBIDDEN_PHRASES)}\n')
for i, phrase in enumerate(FORBIDDEN_PHRASES, 1):
    print(f'  {i:2d}. {phrase}')

## Step 3 — Test blocklist detection (Layer 1)
Feed in texts with known forbidden phrases. The regex should catch them instantly.

In [ ]:
from src.compliance.checker import check_compliance

# These should all be flagged by the blocklist (no LLM call needed)
test_texts_bad = [
    'This serum cures acne in just 7 days!',
    'Clinically proven to reduce wrinkles by 90%.',
    'Guaranteed results or your money back. No side effects whatsoever.',
    'This is a prescription-free cure that replaces your doctor.',
    'Dermatologist approved — complete transformation in 2 weeks.',
]

print('=== BLOCKLIST TESTS (should all be non-compliant) ===\n')
for text in test_texts_bad:
    result = check_compliance(text, use_llm_fallback=False)
    status = '🚨 NON-COMPLIANT' if not result['compliant'] else '✅ COMPLIANT'
    print(f'{status} | Source: {result["source"]}')
    print(f'  Text: "{text}"')
    if result['flagged_phrases']:
        print(f'  Flagged: {result["flagged_phrases"]}')
    print()

## Step 4 — Test clean texts against blocklist
These should pass the blocklist check.

In [ ]:
test_texts_clean = [
    'I really enjoyed using this moisturizer. My skin feels softer.',
    'This retinol serum helped reduce the appearance of fine lines over several months.',
    'I noticed fewer breakouts after switching to this gentle cleanser.',
    'Everyone\'s skin is different — talk to your dermatologist before trying prescription products.',
]

print('=== CLEAN TEXT TESTS (should all pass blocklist) ===\n')
for text in test_texts_clean:
    result = check_compliance(text, use_llm_fallback=False)
    status = '✅ COMPLIANT' if result['compliant'] else '🚨 NON-COMPLIANT'
    print(f'{status} | Source: {result["source"]}')
    print(f'  Text: "{text}"')
    print()

## Step 5 — Test the LLM classifier (Layer 2)
These edge cases dodge the blocklist but contain subtle compliance issues.
The GPT classifier should catch them.

**Note:** This step makes OpenAI API calls.

In [ ]:
edge_cases = [
    'After using this for two weeks my eczema was completely gone and never came back.',
    'Studies show this ingredient eliminates 99% of bacteria that cause skin problems.',
    'You won\'t need your prescription anymore once you start using our natural formula.',
    'I can personally guarantee this will fix your acne — it worked for everyone I know.',
]

print('=== LLM CLASSIFIER TESTS (subtle violations) ===\n')
for text in edge_cases:
    result = check_compliance(text, use_llm_fallback=True)
    status = '🚨 NON-COMPLIANT' if not result['compliant'] else '✅ COMPLIANT'
    print(f'{status} | Source: {result["source"]}')
    print(f'  Text: "{text}"')
    print(f'  Reason: {result.get("reason", "N/A")}')
    if result.get('flagged_phrases'):
        print(f'  Flagged: {result["flagged_phrases"]}')
    print()

## Step 6 — Check real transcript chunks from Pinecone
Pull actual chunks from your corpus and run compliance checks on them.

In [ ]:
from pinecone import Pinecone
from src.ingestion.embedder import embed_texts
from src.utils.config import PINECONE_API_KEY, PINECONE_INDEX_NAME

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

# Retrieve chunks that are most likely to contain claims
query = 'skincare product claims results treatment'
query_vec = embed_texts([query])[0]
results = index.query(vector=query_vec, top_k=5, include_metadata=True)

print(f'=== COMPLIANCE CHECK ON {len(results["matches"])} REAL TRANSCRIPT CHUNKS ===\n')
for i, match in enumerate(results['matches'], 1):
    text = match['metadata'].get('text', '')
    title = match['metadata'].get('title', 'Unknown')
    
    result = check_compliance(text, use_llm_fallback=True)
    status = '🚨 NON-COMPLIANT' if not result['compliant'] else '✅ COMPLIANT'
    
    print(f'--- Chunk {i} (from: {title}) ---')
    print(f'Score: {match["score"]:.4f} | {status} | Source: {result["source"]}')
    print(f'Text (first 200 chars): {text[:200]}...')
    if not result['compliant']:
        print(f'Reason: {result.get("reason", "N/A")}')
        print(f'Flagged: {result.get("flagged_phrases", [])}')
    print()

## Step 7 — Test the agent tool interface
This is how the agent formats compliance results for users.

In [ ]:
from src.compliance.checker import compliance_report

# Non-compliant example
report = compliance_report('This product cures acne and is clinically proven to work.')
print('Non-compliant report:')
print(report)

print('\n' + '='*60 + '\n')

# Compliant example
report = compliance_report('I noticed my skin felt smoother after using this moisturizer for a month.')
print('Compliant report:')
print(report)

## Notes

**Blocklist vs LLM:** The blocklist is free and instant. The LLM classifier costs ~0.001 cents per check but catches edge cases the regex misses.

**Extending the blocklist:** Add new phrases to `FORBIDDEN_PHRASES` in `src/compliance/checker.py` as you discover new patterns.

**False positives:** The LLM classifier may occasionally flag compliant content. Review its reasoning before acting on it.

**Next step:** Move to notebook 04 to build an evaluation dataset and measure quality with LangSmith.